## Setups & Imports

In [35]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import re
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.core.config import settings
pd.set_option("display.max_columns", None)

---

## Decisions (From EDA)

- avoiding leakage: I'll never use `price_egp` to impute `mileage_km`.
- Duplicate definition: ignore `id` and `location`.
- Drop `make` in if it's defined as Other, Unknown.
- Year nulls: extract from `title`, else fill by mode `year` per (make, model), else drop unresolved.
- Year outliers: drop `year < 1975` or `year > 2026`.
- Price filters: drop `price_egp < 20_000` ("Price on call" placeholders) and drop `price_egp > 20_000_000`.
- Mileage rules:
  - drop `mileage_km` nulls
  - drop `mileage_km > 500_000`
  - drop (`year < 2023` AND `mileage_km < 5_000`) for old cars
- Fix obvious categorical inconsistencies before imputation.
- Impute:
  - `fuel`: mode within (make, model, year, transmission) → fallback (make, model) → else drop.
  - `transmission`: mode within (make, model, year, fuel) → fallback (make, model) → else drop.
- Normalize `location` into the 17 categories.
- Rare filter: compute make+model frequency on the final cleaned df, then drop `count < 5`.

---

## Loading the Data

In [36]:
data = settings.load_data("raw")
lookup = settings.load_data("lookup")
data.head(5)

,id,title,make,model,year,mileage_km,transmission,fuel,price_egp,location,scraped_at
0,1,Deepal S 07 2025,Deepal,S 07,2025.0,5000.0,Automatic,hybrid,1650000,"Sheikh Zayed, Giza•",2026-02-20 04:45:52
1,2,Chery Tiggo 2026,Chery,Tiggo,2026.0,NaN,Manual,petrol,915000,NaN,2026-02-20 04:50:40
2,3,Byd F3 2024,BYD,F3,2024.0,4410.0,Manual,petrol,630000,"Ramses + Ramses Extension, Cairo•",2026-02-20 04:41:57
3,4,Byd F3 2025,BYD,F3,2025.0,33033.0,Manual,petrol,610000,"Nasr City, Cairo•",2026-02-20 04:54:05
4,5,Subaru Impreza 2009,Subaru,Impreza,2009.0,98000.0,Manual,petrol,410000,"Maadi, Cairo•",2026-02-20 04:41:57


In [37]:
lookup.head(5)

,make,model,year,engine_cc,horsepower,body_type,drivetrain,seating_capacity,brand_origin,car_segment,brand_market_share
0,Deepal,S 07,2025,1500,318,SUV,AWD,5,chinese,suv,0.0008
1,BYD,F3,2024,1500,109,Sedan,FWD,5,chinese,family,0.0141
2,BYD,F3,2025,1500,109,Sedan,FWD,5,chinese,family,0.0141
3,Subaru,Impreza,2009,1498,107,Sedan,AWD,5,japanese,family,0.0053
4,Peugeot,206,2002,1400,75,Hatchback,FWD,5,european,city,0.0339


In [38]:
lookup.shape

(4190, 11)

In [39]:
data.shape

(26563, 11)

In [40]:
df = data.copy()
df.shape

(26563, 11)

---

## Drop duplicates

In [41]:
dup_all = df.duplicated().sum()

drop_id_cols = [c for c in ["id"] if c in df.columns]
dup_no_id = df.drop(columns=drop_id_cols).duplicated().sum()

drop_id_loc_cols = [c for c in ["id", "location"] if c in df.columns]
dup_no_id_loc = df.drop(columns=drop_id_loc_cols).duplicated().sum()

print("Duplicates (full row):", dup_all)
print("Duplicates (drop id):", dup_no_id)
print("Duplicates (drop id + location):", dup_no_id_loc)

Duplicates (full row): 0
Duplicates (drop id): 32
Duplicates (drop id + location): 65


In [42]:
before_rows = df.shape[0]
dedupe_cols = [c for c in df.columns if c not in {"id", "location"}]
df = df.drop_duplicates(subset=dedupe_cols, keep="first").copy()

print("Rows removed:", before_rows - df.shape[0])
df.shape

Rows removed: 65


(26498, 11)

---

## Categorical standardization + drop Other/Unknown makes

In [43]:
before_rows = df.shape[0]

for col in ["make", "model", "transmission", "fuel", "location", "title"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

if "make" in df.columns:
    make_norm = df["make"].astype("string").str.lower().str.strip()
    df = df[~make_norm.isin(["other", "unknown"])].copy()

if "transmission" in df.columns:
    t_norm = df["transmission"].astype("string").str.lower().str.strip()
    df.loc[t_norm.isin(["0", "0.0", "nan", "none", ""]), "transmission"] = pd.NA

if "fuel" in df.columns:
    f_norm = df["fuel"].astype("string").str.lower().str.strip()
    # normalize known typos (if they appear in future snapshots)
    df.loc[f_norm.isin(["diseal", "diesal"]), "fuel"] = "diesel"

print("Rows removed:", before_rows - df.shape[0])
df.shape

Rows removed: 4


(26494, 11)

---

## Year null recovery (title → mode per make+model → drop)

In [44]:
def extract_year_from_title(title, min_year=1950, max_year=2026):
    if pd.isna(title):
        return pd.NA

    matches = re.findall(r"(?<!\d)(\d{4})(?!\d)", str(title))
    for year_str in reversed(matches):
        year = int(year_str)
        if min_year <= year <= max_year:
            return year

    return pd.NA

rows_before_step = df.shape[0]

df["year"] = pd.to_numeric(df["year"], errors="coerce")
year_nulls_before = int(df["year"].isna().sum())

mask_null_year = df["year"].isna()
df.loc[mask_null_year, "year_from_title"] = df.loc[mask_null_year, "title"].apply(extract_year_from_title)

filled_by_title = int(df.loc[mask_null_year, "year_from_title"].notna().sum())
df.loc[mask_null_year & df["year_from_title"].notna(), "year"] = df.loc[
    mask_null_year & df["year_from_title"].notna(), "year_from_title"
].astype(float)

df.drop(columns=["year_from_title"], inplace=True)

# Fill remaining year nulls from the most frequent year per (make, model)
year_counts = (
    df.loc[df["year"].notna(), ["make", "model", "year"]]
    .groupby(["make", "model", "year"])
    .size()
    .reset_index(name="year_freq")
)

mode_year_per_mm = (
    year_counts.sort_values(
        ["make", "model", "year_freq", "year"],
        ascending=[True, True, False, False],
    )
    .drop_duplicates(["make", "model"])
    .rename(columns={"year": "mode_year_make_model"})
)

df = df.merge(
    mode_year_per_mm[["make", "model", "mode_year_make_model"]],
    on=["make", "model"],
    how="left",
)

mask_fill_mode = df["year"].isna() & df["mode_year_make_model"].notna()
filled_by_mode = int(mask_fill_mode.sum())
df.loc[mask_fill_mode, "year"] = df.loc[mask_fill_mode, "mode_year_make_model"]
df.drop(columns=["mode_year_make_model"], inplace=True)

rows_before_drop = df.shape[0]
df = df[df["year"].notna()].copy()
dropped_unresolved = int(rows_before_drop - df.shape[0])

print("Year nulls before:", year_nulls_before)
print("Filled from title:", filled_by_title)
print("Filled by make+model mode:", filled_by_mode)
print("Dropped unresolved year:", dropped_unresolved)
print("Rows after:", df.shape[0])

df.shape

Year nulls before: 201
Filled from title: 182
Filled by make+model mode: 19
Dropped unresolved year: 0
Rows after: 26494


(26494, 11)

---

## Hard absolute filters (year, price, mileage)

In [45]:
before_rows = df.shape[0]
df = df[(df["year"] >= 1975) & (df["year"] <= 2026)].copy()

print("Dropped year outliers:", before_rows - df.shape[0])
df.shape

Dropped year outliers: 65


(26429, 11)

In [46]:
before_rows = df.shape[0]
df = df[(df["price_egp"] >= 20_000) & (df["price_egp"] <= 20_000_000)].copy()

print("Dropped price outliers (<20k or >20M):", before_rows - df.shape[0])
df.shape

Dropped price outliers (<20k or >20M): 265


(26164, 11)

In [47]:
before_rows = df.shape[0]
df["mileage_km"] = pd.to_numeric(df["mileage_km"], errors="coerce")
df = df[df["mileage_km"].notna()].copy()

print("Dropped null mileage_km:", before_rows - df.shape[0])
df.shape

Dropped null mileage_km: 50


(26114, 11)

In [48]:
before_rows = df.shape[0]
df = df[(df["mileage_km"] >= 0) & (df["mileage_km"] <= 500_000)].copy()

print("Dropped mileage outliers (<0 or >500k):", before_rows - df.shape[0])
df.shape

Dropped mileage outliers (<0 or >500k): 254


(25860, 11)

In [49]:
before_rows = df.shape[0]
old_low_mileage_mask = (df["year"] < 2023) & (df["mileage_km"] < 5_000)
df = df[~old_low_mileage_mask].copy()

print("Dropped (year < 2023 & mileage < 5k):", before_rows - df.shape[0])
df.shape

Dropped (year < 2023 & mileage < 5k): 1573


(24287, 11)

---

## Inconsistency review before imputation

In [50]:
def fix_minor_inconsistencies(
    df: pd.DataFrame,
    col: str,
    group_cols: list[str],
    min_group_size: int = 30,
    min_mode_share: float = 0.98,
    max_minority_count: int = 1,
) -> tuple[pd.DataFrame, pd.DataFrame, int]:
    """
    Fix ultra-rare minority categorical values inside stable groups.

    Guardrails (very strict):
    - group size >= min_group_size
    - mode share >= min_mode_share
    - minority value count <= max_minority_count
    """
    if col not in df.columns:
        return df, pd.DataFrame(), 0

    work = df[df[col].notna()].copy()
    if work.empty:
        return df, pd.DataFrame(), 0

    counts = work.groupby(group_cols + [col]).size().reset_index(name="value_count")
    totals = counts.groupby(group_cols)["value_count"].sum().reset_index(name="group_total")
    counts = counts.merge(totals, on=group_cols, how="left")

    mode_df = (
        counts.sort_values(
            group_cols + ["value_count"],
            ascending=[True] * len(group_cols) + [False],
        )
        .drop_duplicates(group_cols)
        .rename(columns={col: "mode_value", "value_count": "mode_count"})
    )[group_cols + ["mode_value", "mode_count", "group_total"]]
    mode_df["mode_share"] = mode_df["mode_count"] / mode_df["group_total"]

    merged = counts.merge(
        mode_df[group_cols + ["mode_value", "mode_count", "mode_share"]],
        on=group_cols,
        how="left",
    )
    minority = merged[merged[col] != merged["mode_value"]].copy()

    candidates = minority[
        (minority["group_total"] >= min_group_size)
        & (minority["mode_share"] >= min_mode_share)
        & (minority["value_count"] <= max_minority_count)
    ].copy()

    if candidates.empty:
        return df, candidates, 0

    fixed = 0
    out = df.copy()
    for _, row in candidates.iterrows():
        group_mask = pd.Series(True, index=out.index)
        for g in group_cols:
            group_mask &= out[g] == row[g]
        minority_mask = group_mask & (out[col] == row[col])
        fixed += int(minority_mask.sum())
        out.loc[minority_mask, col] = row["mode_value"]

    return out, candidates, fixed


rows_before = df.shape[0]
df, fuel_inconsistency_candidates, fuel_fixed = fix_minor_inconsistencies(
    df,
    col="fuel",
    group_cols=["make", "model", "year"],
    min_group_size=30,
    min_mode_share=0.98,
    max_minority_count=1,
)

df, transmission_inconsistency_candidates, transmission_fixed = fix_minor_inconsistencies(
    df,
    col="transmission",
    group_cols=["make", "model", "year"],
    min_group_size=30,
    min_mode_share=0.98,
    max_minority_count=1,
)

print("Fuel fixes applied:", fuel_fixed)
print("Transmission fixes applied:", transmission_fixed)
print("Fuel candidate groups:", len(fuel_inconsistency_candidates))
print("Transmission candidate groups:", len(transmission_inconsistency_candidates))
print("Rows after:", df.shape[0])

df.shape

Fuel fixes applied: 4
Transmission fixes applied: 0
Fuel candidate groups: 4
Transmission candidate groups: 0
Rows after: 24287


(24287, 11)

---

## Fuel imputation (mode hierarchy)

In [51]:
rows_before = df.shape[0]
fuel_nulls_before = int(df["fuel"].isna().sum())

# Primary: mode fuel within (make, model, year, transmission)
fuel_mode_primary = (
    df.dropna(subset=["fuel", "make", "model", "year", "transmission"])
    .groupby(["make", "model", "year", "transmission"], as_index=False)["fuel"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else pd.NA)
    .rename(columns={"fuel": "fuel_mode_primary"})
)

df = df.merge(
    fuel_mode_primary,
    on=["make", "model", "year", "transmission"],
    how="left",
)

mask_primary = df["fuel"].isna() & df["fuel_mode_primary"].notna()
imputed_primary = int(mask_primary.sum())
df.loc[mask_primary, "fuel"] = df.loc[mask_primary, "fuel_mode_primary"]
df.drop(columns=["fuel_mode_primary"], inplace=True)

# Fallback: mode fuel within (make, model)
fuel_mode_fallback = (
    df.dropna(subset=["fuel", "make", "model"])
    .groupby(["make", "model"], as_index=False)["fuel"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else pd.NA)
    .rename(columns={"fuel": "fuel_mode_fallback"})
)

df = df.merge(fuel_mode_fallback, on=["make", "model"], how="left")
mask_fallback = df["fuel"].isna() & df["fuel_mode_fallback"].notna()
imputed_fallback = int(mask_fallback.sum())
df.loc[mask_fallback, "fuel"] = df.loc[mask_fallback, "fuel_mode_fallback"]
df.drop(columns=["fuel_mode_fallback"], inplace=True)

# Drop anything still unresolved (EDA decision: no match → drop)
rows_before_drop = df.shape[0]
df = df[df["fuel"].notna()].copy()
dropped_unresolved = int(rows_before_drop - df.shape[0])
fuel_nulls_after = int(df["fuel"].isna().sum())

print("Fuel nulls before:", fuel_nulls_before)
print("Imputed (primary mmy+trans):", imputed_primary)
print("Imputed (fallback mm):", imputed_fallback)
print("Dropped unresolved fuel:", dropped_unresolved)
print("Fuel nulls after:", fuel_nulls_after)
print("Rows after:", df.shape[0])

df.shape

Fuel nulls before: 45
Imputed (primary mmy+trans): 12
Imputed (fallback mm): 32
Dropped unresolved fuel: 1
Fuel nulls after: 0
Rows after: 24286


(24286, 11)

---

## Transmission imputation (mode hierarchy)

In [52]:
rows_before = df.shape[0]
trans_nulls_before = int(df["transmission"].isna().sum())

# Primary: mode transmission within (make, model, year, fuel)
trans_mode_primary = (
    df.dropna(subset=["transmission", "make", "model", "year", "fuel"])
    .groupby(["make", "model", "year", "fuel"], as_index=False)["transmission"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else pd.NA)
    .rename(columns={"transmission": "trans_mode_primary"})
)

df = df.merge(trans_mode_primary, on=["make", "model", "year", "fuel"], how="left")
mask_primary = df["transmission"].isna() & df["trans_mode_primary"].notna()
imputed_primary = int(mask_primary.sum())
df.loc[mask_primary, "transmission"] = df.loc[mask_primary, "trans_mode_primary"]
df.drop(columns=["trans_mode_primary"], inplace=True)

# Fallback: mode transmission within (make, model)
trans_mode_fallback = (
    df.dropna(subset=["transmission", "make", "model"])
    .groupby(["make", "model"], as_index=False)["transmission"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else pd.NA)
    .rename(columns={"transmission": "trans_mode_fallback"})
)

df = df.merge(trans_mode_fallback, on=["make", "model"], how="left")
mask_fallback = df["transmission"].isna() & df["trans_mode_fallback"].notna()
imputed_fallback = int(mask_fallback.sum())
df.loc[mask_fallback, "transmission"] = df.loc[mask_fallback, "trans_mode_fallback"]
df.drop(columns=["trans_mode_fallback"], inplace=True)

# Drop anything still unresolved
rows_before_drop = df.shape[0]
df = df[df["transmission"].notna()].copy()
dropped_unresolved = int(rows_before_drop - df.shape[0])
trans_nulls_after = int(df["transmission"].isna().sum())

print("Transmission nulls before:", trans_nulls_before)
print("Imputed (primary mmy+fuel):", imputed_primary)
print("Imputed (fallback mm):", imputed_fallback)
print("Dropped unresolved transmission:", dropped_unresolved)
print("Transmission nulls after:", trans_nulls_after)
print("Rows after:", df.shape[0])

df.shape

Transmission nulls before: 30
Imputed (primary mmy+fuel): 29
Imputed (fallback mm): 1
Dropped unresolved transmission: 0
Transmission nulls after: 0
Rows after: 24286


(24286, 11)

---

## Location normalization (from EDA)

In [53]:
def normalize_locations(df: pd.DataFrame, col: str = "location") -> pd.DataFrame:
    """
    Normalize raw Egyptian car marketplace location strings into
    clean, model-ready location categories.

    Strategy:
    - Hierarchical np.select: specific neighborhoods checked before broad cities
    - Sparse governorates (< 300 rows each) consolidated into 'Delta (Other)'
    - Anything unmatched → 'Other/Unknown' (includes raw '•' placeholders)
    - Raw location column is NOT dropped here — caller decides what to keep

    Final categories (17):
        New Cairo, October & Zayed, Nasr City, Heliopolis, Maadi,
        Alexandria, Cairo, Giza, Qalyubia,
        Sharqia, Dakahlia, Gharbia, Canal Zone,
        Delta (Other) [= Monufia + Kafr el-Sheikh + Beheira + Damietta],
        Upper Egypt, Coastal & Resorts, Other/Unknown

    Parameters
    ----------
    df  : DataFrame containing the raw location column
    col : Name of the raw location column (default: 'location')

    Returns
    -------
    location column fixed and normalized
    """

    # ── 1. Pre-clean ────────────────────────────────────────────────────────
    # Convert NaN to "nan" string so str methods work uniformly.
    # Strip all non-alphanumeric characters (bullet •, comma, dash, etc.)
    clean_s = (
        df[col]
        .astype(str)
        .str.lower()
        .str.replace(r"[^a-z0-9\s]", " ", regex=True)
        .str.strip()
    )

    # ── 2. Conditions (ORDER MATTERS — specific before general) ─────────────
    conditions = [

        # ── TIER 1: New Cairo cluster (East Cairo) ──────────────────────────
        # 'badr' intentionally excluded — Badr City is 80 km east, not New Cairo
        clean_s.str.contains(
            r"tagamo|new cairo|settlement|rehab|madinaty|mostakbal|shorouk|katameya",
            na=False,
        ),

        # ── TIER 2: October & Zayed cluster (West Cairo) ────────────────────
        clean_s.str.contains(r"october|zayed", na=False),

        # ── TIER 3: Major Cairo districts ───────────────────────────────────
        # Checked before broad 'cairo' to avoid misclassification
        clean_s.str.contains(r"nasr city", na=False),

        clean_s.str.contains(
            r"heliopolis|masr el gedida|masr al jadidah|sheraton",
            na=False,
        ),

        clean_s.str.contains(r"maadi|basatin|tora", na=False),

        # ── TIER 4: Alexandria + western suburbs ────────────────────────────
        # 'agami' AND 'agamy' — both spellings appear in the data
        # 'amreya'/'amereyah' — Alexandria western district
        # 'borg al arab' — Alexandria new city
        clean_s.str.contains(
            r"alexandria|agami|agamy|smoha|smouha|sidi gaber|gaber"
            r"|beshr|asafra|miami|borg al arab|amreya|amereyah",
            na=False,
        ),

        # ── TIER 5: Cairo (broad catchall) ──────────────────────────────────
        # Covers: Helwan, Mokattam, Shubra, Ain Shams, Matareya, Marg,
        #         Obour City (listed as Cairo), Badr City (listed as Cairo),
        #         Rod al-Farag, Salam City, downtown districts
        # Note: 'shubra' is here AND Shubra al-Khaimah is in Qalyubia below.
        #       np.select takes first match — Cairo checked first, but
        #       'shubra al khaymah'/'shubra al khaimah' won't match 'shubra'
        #       alone only if you match exact. To be safe, Qalyubia regex
        #       uses full 'shubra al khaymah' pattern.
        clean_s.str.contains(
            r"cairo|mokattam|helwan|downtown|shubra|nozha|nozah"
            r"|badr city|obour|salam city|ain shams|matareya"
            r"|marg|rod al farag|hadayek al kobba|abidin|gesr al suez",
            na=False,
        ),

        # ── TIER 6: Giza ────────────────────────────────────────────────────
        # 'mariotya' and 'mariotaya' — Maryoteyya district in Giza
        # 'hadayek al ahram' — Pyramids Gardens area in Giza
        # 'ramses' — Ramses area borders Giza
        clean_s.str.contains(
            r"giza|haram|faisal|dokki|imbaba|mohandessin|pyramids"
            r"|mariotaya|maryotaya|mariotya|hadayek al ahram|hadayek october"
            r"|ramses",
            na=False,
        ),

        # ── TIER 7: Qalyubia ────────────────────────────────────────────────
        # Full 'shubra al khaymah' pattern catches Shubra al-Khaimah correctly
        # 'obour city' also listed as Qalyubia in some entries
        clean_s.str.contains(
            r"qalyubia|banha|shubra al khaymah|shubra al khaimah|shubrakhit",
            na=False,
        ),

        # ── TIER 8: Sharqia ─────────────────────────────────────────────────
        # '10th of ramadan' is in Sharqia governorate — some listings
        # incorrectly say 'Cairo'; Sharqia condition checked before Cairo
        # so they are classified correctly
        clean_s.str.contains(
            r"sharqia|zagazig|10th of ramadan|bilbeis",
            na=False,
        ),

        # ── TIER 9: Dakahlia ────────────────────────────────────────────────
        # 'mansura' and 'mansoura' — common spelling variants
        clean_s.str.contains(r"dakahlia|mansoura|mansura", na=False),

        # ── TIER 10: Gharbia ────────────────────────────────────────────────
        clean_s.str.contains(r"gharbia|tanta|mahalla", na=False),

        # ── TIER 11: Monufia → will be consolidated into Delta (Other) ──────
        clean_s.str.contains(r"monufia|menofia|menufeya|shebin|sadat", na=False),

        # ── TIER 12: Kafr el-Sheikh → Delta (Other) ─────────────────────────
        # 'kafr el cheik' — common misspelling found in the data
        clean_s.str.contains(
            r"kafr el sheikh|kafr al sheikh|kafr el cheik|kafr elsheikh",
            na=False,
        ),

        # ── TIER 13: Beheira → Delta (Other) ────────────────────────────────
        # 'el behiera' / 'behiera' — misspelling found in the data
        clean_s.str.contains(
            r"beheira|behiera|damanhour|kafr el dawwar|kafr al dawwar",
            na=False,
        ),

        # ── TIER 14: Damietta → Delta (Other) ───────────────────────────────
        clean_s.str.contains(r"damietta", na=False),

        # ── TIER 15: Canal Zone ─────────────────────────────────────────────
        # Ismailia, Port Said, Suez, Ain Sokhna
        clean_s.str.contains(r"ismailia|port said|suez|ain sokhna", na=False),

        # ── TIER 16: Upper Egypt ────────────────────────────────────────────
        # 'el wadi el gedid' / 'new valley' — remote western governorate
        clean_s.str.contains(
            r"fayoum|faiyum|beni suef|ben suef|minya|asyut|sohag"
            r"|qena|luxor|aswan|kom ombo|wadi el gedid|wadi gedid|new valley",
            na=False,
        ),

        # ── TIER 17: Coastal & Resorts ──────────────────────────────────────
        # Red Sea, North Coast, Sinai
        clean_s.str.contains(
            r"red sea|hurghada|gouna|matruh|mersa matruh|marsa matrouh"
            r"|north coast|sahel|sinai|sharm",
            na=False,
        ),
    ]

    choices = [
        "New Cairo",        # Tier 1
        "October & Zayed",  # Tier 2
        "Nasr City",        # Tier 3a
        "Heliopolis",       # Tier 3b
        "Maadi",            # Tier 3c
        "Alexandria",       # Tier 4
        "Cairo",            # Tier 5
        "Giza",             # Tier 6
        "Qalyubia",         # Tier 7
        "Sharqia",          # Tier 8
        "Dakahlia",         # Tier 9
        "Gharbia",          # Tier 10
        "Monufia",          # Tier 11 → consolidated below
        "Kafr el-Sheikh",   # Tier 12 → consolidated below
        "Beheira",          # Tier 13 → consolidated below
        "Damietta",         # Tier 14 → consolidated below
        "Canal Zone",       # Tier 15
        "Upper Egypt",      # Tier 16
        "Coastal & Resorts",# Tier 17
    ]

    # ── 3. Apply selection ─────────────────────────────────────────────────
    df["location"] = np.select(conditions, choices, default="Other/Unknown")

    # ── 4. Handle actual nulls and placeholder-only strings ─────────────────
    # After .astype(str), original NaN → "nan"; bullet-only rows → "  " etc.
    df.loc[df[col].isna(), "location"] = "Other/Unknown"
    df.loc[clean_s.isin(["", "nan", "none"]), "location"] = "Other/Unknown"

    # ── 5. Consolidate sparse governorates into Delta (Other) ───────────────
    # Each of these has < 350 rows — too sparse for reliable independent signal.
    # Combined they form a meaningful 'Delta region (other)' group (~940 rows).
    sparse_governorates = {"Monufia", "Kafr el-Sheikh", "Beheira", "Damietta"}
    df["location"] = df["location"].replace(
        {gov: "Delta (Other)" for gov in sparse_governorates},
    )

    return df["location"]


# ── Final category reference ───────────────────────────────────────────────
LOCATION_CATEGORIES = [
    # Greater Cairo cluster (~65% of listings)
    "New Cairo",         # 19.7% — Tagamo3, 5th Settlement, Rehab, Shorouk
    "Cairo",             # 18.5% — Helwan, Mokattam, Shubra, Ain Shams, etc.
    "October & Zayed",   # 10.2% — 6 October, Sheikh Zayed
    "Nasr City",         #  9.0% — Cairo district, distinct market
    "Giza",              #  8.2% — Haram, Faisal, Dokki, Mohandessin
    "Heliopolis",        #  5.3% — Masr el Gedida, Sheraton
    "Maadi",             #  2.6% — Maadi, Basatin, Tora

    # Other major cities
    "Alexandria",        #  7.4% — All Alexandria + western suburbs

    # Delta governorates (each 2-2.5%)
    "Sharqia",           #  2.3%
    "Gharbia",           #  2.3%
    "Dakahlia",          #  2.0%
    "Qalyubia",          #  1.5%
    "Delta (Other)",     #  3.6% — Monufia + Kafr el-Sheikh + Beheira + Damietta

    # Other regions
    "Canal Zone",        #  1.8% — Ismailia, Port Said, Suez
    "Upper Egypt",       #  3.0% — Minya, Asyut, Sohag, Luxor, Aswan, etc.
    "Coastal & Resorts", #  1.0% — Hurghada, North Coast, Sinai, Matrouh

    # Fallback
    "Other/Unknown",     #  1.6% — Raw '•' placeholders + unmatched strings
]

df["location"] = normalize_locations(df, col="location")

df["location"].value_counts().to_frame().head(30)


,count
location,
New Cairo,5009
Cairo,4204
October & Zayed,2479
Nasr City,2203
Giza,1928
Alexandria,1779
Heliopolis,1328
Delta (Other),824
Other/Unknown,725


---

## Rare make+model filter computed on the cleaned df

In [54]:
before_rows = df.shape[0]
df["make_model"] = df["make"].astype("string") + " " + df["model"].astype("string")

mm_counts = df["make_model"].value_counts()
rare_mm = mm_counts[mm_counts < 5].index
rare_rows = int(df["make_model"].isin(rare_mm).sum())
print("Rows that are rare (make+model count < 5):", rare_rows)

df = df[~df["make_model"].isin(rare_mm)].copy()
df.drop(columns=["make_model"], inplace=True)

print("Rows removed:", before_rows - df.shape[0])
df.shape

Rows that are rare (make+model count < 5): 1034
Rows removed: 1034


(23252, 11)

In [55]:
df.head()

,id,title,make,model,year,mileage_km,transmission,fuel,price_egp,location,scraped_at
0,1,Deepal S 07 2025,Deepal,S 07,2025.0,5000.0,Automatic,hybrid,1650000,October & Zayed,2026-02-20 04:45:52
1,3,Byd F3 2024,BYD,F3,2024.0,4410.0,Manual,petrol,630000,Cairo,2026-02-20 04:41:57
2,4,Byd F3 2025,BYD,F3,2025.0,33033.0,Manual,petrol,610000,Nasr City,2026-02-20 04:54:05
3,5,Subaru Impreza 2009,Subaru,Impreza,2009.0,98000.0,Manual,petrol,410000,Maadi,2026-02-20 04:41:57
4,6,Byd F3 2024,BYD,F3,2024.0,78642.0,Manual,petrol,570000,Cairo,2026-02-20 04:41:57


---

## Log transform the price

In [56]:
df['price_egp_log'] = np.log10(df['price_egp'])

---

## Engineered features

In [57]:
df['car_age'] = 2026 - df['year']

In [58]:
df['mileage_per_year'] = df['mileage_km'] / df['car_age'].replace(0, 1)

In [59]:
df.head(20)

,id,title,make,model,year,mileage_km,transmission,fuel,price_egp,location,scraped_at,price_egp_log,car_age,mileage_per_year
0,1,Deepal S 07 2025,Deepal,S 07,2025.0,5000.0,Automatic,hybrid,1650000,October & Zayed,2026-02-20 04:45:52,6.217484,1.0,5000.000000
1,3,Byd F3 2024,BYD,F3,2024.0,4410.0,Manual,petrol,630000,Cairo,2026-02-20 04:41:57,5.799341,2.0,2205.000000
2,4,Byd F3 2025,BYD,F3,2025.0,33033.0,Manual,petrol,610000,Nasr City,2026-02-20 04:54:05,5.785330,1.0,33033.000000
3,5,Subaru Impreza 2009,Subaru,Impreza,2009.0,98000.0,Manual,petrol,410000,Maadi,2026-02-20 04:41:57,5.612784,17.0,5764.705882
4,6,Byd F3 2024,BYD,F3,2024.0,78642.0,Manual,petrol,570000,Cairo,2026-02-20 04:41:57,5.755875,2.0,39321.000000
5,7,Peugeot 206 2002,Peugeot,206,2002.0,200000.0,Manual,petrol,230000,Alexandria,2026-02-20 04:41:57,5.361728,24.0,8333.333333
6,8,Peugeot 405 1996,Peugeot,405,1996.0,200000.0,Manual,petrol,130000,Alexandria,2026-02-20 04:41:57,5.113943,30.0,6666.666667
8,10,Mercedes C200 AMG Wakeel only 45k km 2020,Mercedes,C200,2020.0,45000.0,Manual,petrol,2300000,New Cairo,2026-02-20 04:41:57,6.361728,6.0,7500.000000
9,11,Chana Benni 2009,Chana,Benni,2009.0,126000.0,Manual,petrol,150000,New Cairo,2026-02-20 04:41:57,5.176091,17.0,7411.764706
10,12,Fiat 131 1981,Fiat,131,1981.0,100000.0,Manual,petrol,72000,Giza,2026-02-20 04:41:57,4.857332,45.0,2222.222222


---

## Final checks & save cleaned data

In [60]:
REQUIRED_COLS = [
    "make",
    "model",
    "year",
    "mileage_km",
    "transmission",
    "fuel",
    "price_egp",
    "location",
]

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
assert not missing_cols, f"Missing required columns: {missing_cols}"

null_counts = df[REQUIRED_COLS].isna().sum()
bad_nulls = null_counts[null_counts > 0]
assert bad_nulls.empty, f"Nulls remain in required columns:\n{bad_nulls}"

# ── Hard rules from EDA decisions ────────────────────────────────────────────
assert df["year"].between(1975, 2026).all(), "Year out of bounds (1975–2026)"
assert (df["price_egp"] >= 20_000).all(), "price_egp below 20k"
assert (df["price_egp"] <= 20_000_000).all(), "price_egp above 20M"
assert df["mileage_km"].between(0, 500_000).all(), "mileage_km out of bounds (0–500k)"
assert (~((df["year"] < 2023) & (df["mileage_km"] < 5_000))).all(), "Found (year < 2023 & mileage < 5k)"
assert (~df["make"].isin(["Other", "Unknown"])).all(), "Found make in {Other, Unknown}"

# Location categories should be only from the EDA mapping
bad_locations = set(df["location"].dropna().unique()) - set(LOCATION_CATEGORIES)
assert not bad_locations, f"Unexpected location categories: {sorted(bad_locations)}"


In [61]:
df["year"] = df["year"].astype(int)
df["mileage_km"] = df["mileage_km"].astype(int)
df["price_egp"] = df["price_egp"].astype(int)

In [62]:
df = df.drop(columns=["id", "title", "scraped_at"]).copy()
lookup = lookup.drop(columns=["brand_market_share"]).copy()
print("Final cleaned shape:", df.shape)

Final cleaned shape: (23252, 11)


In [63]:
df.head()

,make,model,year,mileage_km,transmission,fuel,price_egp,location,price_egp_log,car_age,mileage_per_year
0,Deepal,S 07,2025,5000,Automatic,hybrid,1650000,October & Zayed,6.217484,1.0,5000.000000
1,BYD,F3,2024,4410,Manual,petrol,630000,Cairo,5.799341,2.0,2205.000000
2,BYD,F3,2025,33033,Manual,petrol,610000,Nasr City,5.785330,1.0,33033.000000
3,Subaru,Impreza,2009,98000,Manual,petrol,410000,Maadi,5.612784,17.0,5764.705882
4,BYD,F3,2024,78642,Manual,petrol,570000,Cairo,5.755875,2.0,39321.000000


In [64]:
settings.save_data(df, "cleaned")
# unique_main_info = df[["make", "model", "year", "transmission", "fuel"]].drop_duplicates()
# settings.save_data(unique_main_info, "main_info", format="all")

print("Data shape before cleaning:", data.shape)
print("Final cleaned shape:", df.shape)
print(f"Total rows removed: {data.shape[0] - df.shape[0]} ({(data.shape[0] - df.shape[0]) / data.shape[0]:.1%})")


Data shape before cleaning: (26563, 11)
Final cleaned shape: (23252, 11)
Total rows removed: 3311 (12.5%)


## Merge the data with lookup

In [65]:
df = df.merge(lookup, on=["make", "model", "year"], how="left")

In [66]:
final_cols = [

    "make",
    "model",
    "year",
    "car_age",
    "transmission",
    "fuel",
    "mileage_km",
    "mileage_per_year",
    "location",
    "engine_cc",
    "horsepower",
    "body_type",
    "drivetrain",
    "seating_capacity",
    "brand_origin",
    "car_segment",
    "price_egp",
    "price_egp_log",
]

df = df[final_cols].copy()
df.head()

,make,model,year,car_age,transmission,fuel,mileage_km,mileage_per_year,location,engine_cc,horsepower,body_type,drivetrain,seating_capacity,brand_origin,car_segment,price_egp,price_egp_log
0,Deepal,S 07,2025,1.0,Automatic,hybrid,5000,5000.000000,October & Zayed,1500,318,SUV,AWD,5,chinese,suv,1650000,6.217484
1,BYD,F3,2024,2.0,Manual,petrol,4410,2205.000000,Cairo,1500,109,Sedan,FWD,5,chinese,family,630000,5.799341
2,BYD,F3,2025,1.0,Manual,petrol,33033,33033.000000,Nasr City,1500,109,Sedan,FWD,5,chinese,family,610000,5.785330
3,Subaru,Impreza,2009,17.0,Manual,petrol,98000,5764.705882,Maadi,1498,107,Sedan,AWD,5,japanese,family,410000,5.612784
4,BYD,F3,2024,2.0,Manual,petrol,78642,39321.000000,Cairo,1500,109,Sedan,FWD,5,chinese,family,570000,5.755875


In [67]:
df.shape

(23252, 18)

In [68]:
# save final cleaned and feature-enriched dataset
settings.save_data(df, "processed", format="csv")